# 00 · Los datos y el problema

**Taller 3 de RL (MIAX) · Ejecución de una orden CARE de compra con Repsol.**

Antes de entrenar nada, este notebook responde a tres preguntas: **qué datos tenemos**, **qué tiene que decidir el agente**
y **cuánto cuesta hacerlo bien o mal**. Al final quedan medidas las referencias que usan los cuatro ejercicios.

| Sección | Pregunta |
|---|---|
| 1 | ¿Qué hay en los datos? |
| 2 | ¿Cómo es un día de mercado? |
| 3 | ¿Qué pasa en todos los días? Volumen, spread y volatilidad |
| 4 | ¿Qué es un episodio y qué ve el agente? |
| 5 | ¿Cuánto cuesta comprar? VWAP y spread |
| 6 | ¿Funciona el entorno? Invariantes |
| 7 | Referencias: TWAP, aleatorio y casos extremos |

## 0. El problema en una frase

> Un cliente nos pide comprar **n títulos** en las próximas **N ventanas de 5 s** (entre 33 y 50 minutos). En cada ventana
> decidimos **comprar 1 título al ask** o **esperar**. Queremos completar la orden pagando lo menos posible frente al
> **VWAP del mercado** en ese intervalo.

Sin impacto de mercado ni costes (simplificación del enunciado). El ej. 2 pide que el agente aprenda a repartir las compras
**como un TWAP**: una cada N/n ventanas, en línea recta.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath("."))   # ejecutar desde notebooks/
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from common import data as dt, rewards as rw, protocol as pr, plots as pl
from common.env import BestExecutionEnv
from common.agents import TWAPAgent
pd.set_option("display.precision", 3)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3, "axes.spines.top": False, "axes.spines.right": False})
C_MKT, C_ALGO, C_MID = pl.MKT, pl.ALGO, pl.MID
%matplotlib inline

## 1. Qué hay en los datos

`rep_data.pkl` es un diccionario `{train, val, test}` con **un DataFrame por día**: una fila cada 5 s entre las 13:00 y
las 17:25, con el primer nivel del libro.

| Columna | Significado |
|---|---|
| `ask1` / `bid1` | Mejor precio de venta / de compra |
| `vask1` / `vbid1` | Títulos disponibles a ese precio |
| `last` | Último precio negociado |
| `cumvol` | Volumen acumulado del día (la diferencia entre filas = títulos negociados en esos 5 s) |

In [ ]:
DATA = dt.load("REP")
rows = []
for split, days in DATA.items():
    keys = list(days)
    first, last = days[keys[0]], days[keys[-1]]
    rows.append({"split": split, "días": len(keys), "desde": first.index[0].date(), "hasta": last.index[0].date(),
                 "filas por día": int(np.median([len(d) for d in days.values()])),
                 "horario": f"{first.index[0]:%H:%M}–{first.index[-1]:%H:%M}"})
pd.DataFrame(rows).set_index("split")

In [ ]:
train = DATA["train"]
day0 = list(train)[0]
train[day0].head()

**Lectura:** según el notebook del taller, train son los 60 primeros días hábiles de 2018 y val y test, 30 días cada uno de
2019. 4 h 25 min cada 5 s dan unas 3.180 filas por día. Trabajamos con **train**: el enunciado pide las figuras en train y no
exige validación.

## 2. Un día de mercado

Arriba, el precio medio (mid) con el spread. Abajo, el volumen negociado en cada ventana de 5 s y el acumulado del día.

In [ ]:
df = train[day0]
mid = (df.ask1 + df.bid1) / 2
vol = df.cumvol.diff().clip(lower=0).fillna(0)
fig, ax = plt.subplots(3, 1, figsize=(13, 8), sharex=True, gridspec_kw={"height_ratios": [2.2, 1, 1]})
ax[0].fill_between(df.index, df.bid1, df.ask1, color=C_MKT, alpha=0.25, label="spread (bid–ask)")
ax[0].plot(mid, color=C_MKT, lw=0.8, label="mid")
ax[0].set_title(f"Repsol · {df.index[0]:%d/%m/%Y}"); ax[0].legend(loc="upper left"); ax[0].set_ylabel("€")
ax[1].bar(df.index, vol, width=4/86400, color=C_MKT, alpha=0.6); ax[1].set_ylabel("títulos / 5 s")
ax[2].plot(df.cumvol - df.cumvol.iloc[0], color=C_MKT); ax[2].set_ylabel("vol. acumulado")
plt.tight_layout()

In [ ]:
zoom = df.iloc[1200:1260]   # 5 minutos
fig, ax = plt.subplots(figsize=(13, 3))
ax.step(zoom.index, zoom.ask1, where="post", color=C_MID, label="ask1 (aquí compra el agente)")
ax.step(zoom.index, zoom.bid1, where="post", color=C_MKT, label="bid1")
ax.step(zoom.index, (zoom.ask1 + zoom.bid1) / 2, where="post", color="k", lw=0.8, ls="--", label="mid (precio de mercado del entorno)")
ax.set_title("Zoom de 5 minutos: el precio se mueve a saltos de tick"); ax.legend(fontsize=8)
plt.tight_layout()

**Qué mirar:**
- El precio se mueve **a saltos de tick**, y el spread es la distancia entre las dos escaleras. ¿Cuántos ticks suele medir?
- El volumen llega **a ráfagas**: ventanas de 5 s sin operaciones junto a otras con miles de títulos.

## 3. Todos los días: volumen, spread y volatilidad a lo largo de la sesión

Promedio de los 60 días de train en tramos de 5 minutos.

In [ ]:
frames = []
for d in train.values():
    m = (d.ask1 + d.bid1) / 2
    frames.append(pd.DataFrame({
        "hora": d.index.floor("5min").strftime("%H:%M"),
        "vol": d.cumvol.diff().clip(lower=0).fillna(0),
        "spread_bps": (d.ask1 - d.bid1) / m * 1e4,
        "ret_bps": np.log(m).diff().fillna(0) * 1e4,
        "zero": (d.cumvol.diff().fillna(0) == 0),
    }))
allx = pd.concat(frames)
prof = allx.groupby("hora").agg(vol=("vol", "sum"), spread=("spread_bps", "mean"), vola=("ret_bps", "std"), zero=("zero", "mean"), filas=("vol", "size"))
prof = prof[prof.filas >= prof.filas.median() / 2]   # quita el tramo final incompleto (17:25)
prof["vol"] /= len(train)
fig, ax = plt.subplots(2, 2, figsize=(13, 6.5), sharex=True)
for a, col, title, color in [(ax[0, 0], "vol", "volumen medio por tramo de 5 min (títulos)", C_MKT),
                             (ax[0, 1], "spread", "spread medio (pb)", C_MID),
                             (ax[1, 0], "vola", "volatilidad del mid en 5 s (pb)", "#6B46C1"),
                             (ax[1, 1], "zero", "fracción de ventanas de 5 s sin negociación", "#718096")]:
    a.plot(prof.index, prof[col], color=color); a.set_title(title)
for a in ax[1]:
    a.set_xticks(prof.index[::6]); a.tick_params(axis="x", rotation=45)
for a in ax.ravel():
    a.axvline("15:30", color="k", ls=":", lw=1)
fig.suptitle("Perfil intradía · línea de puntos: 15:30, apertura de EE. UU."); plt.tight_layout()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 3.5))
ax[0].hist(allx.spread_bps.clip(upper=20), bins=60, color=C_MID, alpha=0.7)
ax[0].set_title(f"spread (pb): mediana {allx.spread_bps.median():.1f}"); ax[0].set_xlabel("pb")
daily = pd.Series({k: v.cumvol.iloc[-1] - v.cumvol.iloc[0] for k, v in train.items()})
ax[1].bar(range(len(daily)), daily.values / 1e6, color=C_MKT, alpha=0.7)
ax[1].set_title("volumen negociado de 13:00 a 17:25 por día (millones)"); ax[1].set_xlabel("día de train")
plt.tight_layout()

**Qué mirar:**
- **¿Es uniforme el volumen?** Si sube hacia el cierre o con la apertura americana (15:30), el TWAP, que reparte por tiempo y
  no por volumen, comprará poco donde el mercado negocia mucho, y su VWAP se separará del de mercado.
- **El spread en pb** es el coste mínimo de comprar agresivo: el agente siempre cruza el spread.
- **La fracción de ventanas sin negociación:** cuanto mayor sea, más dominan el VWAP de un episodio unas pocas ventanas con mucho volumen.

## 4. Qué es un episodio y qué ve el agente

Cada `reset()` sortea una orden nueva:
- **Día** de train y **hora de inicio** al azar.
- **Duración N** ∈ [400, 600] ventanas de 5 s, es decir, entre 33 y 50 minutos.
- **Tamaño n** = ρ · N con ρ ∈ [7,5 %, 12,5 %]: hay que comprar en ~1 de cada 10 ventanas.
- 60 ventanas previas (`look_back`) quedan disponibles para calcular características del mercado.

El agente solo ve **s = [T, V]**: el tiempo que queda y el volumen que queda, en tanto por uno.

In [ ]:
env = BestExecutionEnv(train, seed=0)
eps = []
for _ in range(2000):
    env.reset(); eps.append((env.episode_bins, env.vol_care, env.episode.index[0].strftime("%H:%M")))
eps = pd.DataFrame(eps, columns=["N", "n", "inicio"])
fig, ax = plt.subplots(1, 3, figsize=(13, 3.3))
ax[0].hist(eps.N, bins=30, color=C_MKT, alpha=0.7); ax[0].set_title("duración N (ventanas)")
ax[1].hist(eps.n, bins=range(eps.n.min(), eps.n.max() + 2), color=C_ALGO, alpha=0.7); ax[1].set_title("tamaño de la orden n (títulos)")
ax[2].hist(pd.to_datetime(eps.inicio, format="%H:%M").dt.hour + pd.to_datetime(eps.inicio, format="%H:%M").dt.minute / 60,
           bins=24, color="#718096", alpha=0.7); ax[2].set_title("hora de inicio")
plt.tight_layout()
eps.describe().loc[["min", "mean", "max"]]

In [ ]:
env.reset(day=day0, start_pos=1500, n_bins=500, vol_care=50)
full = train[day0]
m = (full.ask1 + full.bid1) / 2
fig, ax = plt.subplots(figsize=(13, 3.2))
ax.plot(m, color="#A0AEC0", lw=0.7, label="mid del día")
ax.axvspan(env.episode_full.index[0], env.episode.index[0], color="#718096", alpha=0.25, label="look_back (60 ventanas)")
ax.axvspan(env.episode.index[0], env.episode.index[-1], color=C_ALGO, alpha=0.25, label="episodio: N = 500, n = 50")
ax.set_title("Un episodio dentro de su día"); ax.legend(loc="upper left", fontsize=8)
plt.tight_layout()

**El bucle de un paso.** `step(a)`: calcula el VWAP de mercado hasta ese momento → ejecuta la acción (compra 1 al `ask1` o
nada) → calcula la recompensa → avanza `state_pos` → comprueba si ha terminado. El episodio termina cuando **se completa la
orden** o **se acaba el tiempo** (con volumen pendiente).

En el plano (T, V) un episodio empieza en (1, 1) y termina cuando V = 0 o T = 0. El TWAP perfecto es la **diagonal**: compra
cuando va igual o por detrás del plan (V ≥ T).

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
for agent_name, color, act in [("TWAP determinista", C_ALGO, lambda s, e: int(s[1] >= s[0] - 1e-9)),
                               ("comprar todo al principio", C_MID, lambda s, e: 1),
                               ("aleatorio (p = ρ)", "#718096", lambda s, e: e.action_sample())]:
    e = BestExecutionEnv(train, seed=3); s = e.reset(day=day0, start_pos=1500, n_bins=500, vol_care=50)
    S = [s]
    while True:
        s, _, done, _ = e.step(act(S[-1], e))
        if done: break
        S.append(s)
    S = np.array(S)
    ax[0].plot(S[:, 0], S[:, 1], color=color, lw=1.4, label=agent_name)
    ax[1].plot(np.arange(len(e.vol_hist)) / e.episode_bins, np.cumsum(e.vol_hist) / e.vol_care, color=color, lw=1.4, label=agent_name)
ax[0].plot([0, 1], [0, 1], "k--", lw=1); ax[0].invert_xaxis()
ax[0].set_xlabel("tiempo restante T  (el tiempo avanza →)"); ax[0].set_ylabel("volumen restante V"); ax[0].set_title("Lo que ve el agente: estados (T, V)")
ax[1].plot([0, 1], [0, 1], "k--", lw=1, label="plan TWAP")
ax[1].set_xlabel("tiempo transcurrido t/N"); ax[1].set_ylabel("volumen ejecutado e/n"); ax[1].set_title("La misma trayectoria en el tiempo")
ax[0].legend(fontsize=8); ax[1].legend(fontsize=8); plt.tight_layout()

## 5. Cuánto cuesta comprar: VWAP y spread

La métrica del caso es el **VWAP slippage** frente al mercado, en puntos básicos:

$$\text{slippage} = \frac{\text{VWAP}_{\text{mercado}} - \text{VWAP}_{\text{algo}}}{\text{VWAP}_{\text{mercado}}}\cdot 10^4 \qquad (>0 \text{: compramos más barato que el mercado})$$

El entorno valora el mercado al **mid**, pero el agente compra al **ask**. Por tanto, aunque compre en los mismos momentos
que el mercado, **paga medio spread de más**: el slippage esperado de cualquier estrategia agresiva ronda −½ spread.

In [ ]:
half = allx.spread_bps.median() / 2
print(f"Medio spread mediano: {half:.2f} pb  →  slippage esperado de comprar agresivo ≈ -{half:.2f} pb")

## 6. ¿Funciona el entorno? Invariantes

16 pruebas sobre datos sintéticos con el mismo esquema: rangos de N y n, VWAP calculado a mano, fin por orden completa o por
tiempo, el TWAP determinista con error ≤ 1 título, recompensas, checkpoints y ε.

In [ ]:
import unittest
unittest.main(module="common.test_env", argv=["x"], exit=False, verbosity=1)

## 7. Referencias

Cuatro políticas sin aprendizaje sobre la **rejilla fija** de evaluación (27 órdenes: N ∈ {400, 500, 600} × ρ ∈ {7,5 %, 10 %,
12,5 %} × 3 días):

| Política | Regla |
|---|---|
| TWAP determinista | Compra si va igual o por detrás del plan (V ≥ T) |
| Aleatoria | Compra con probabilidad ρ |
| Todo al principio | Compra en cada ventana hasta acabar |
| Todo al final | Espera y compra en las últimas n ventanas |

Métricas: **error máximo de seguimiento** frente al plan TWAP (en títulos), **instante de finalización** (t/N), **volumen
pendiente** y **slippage** frente al VWAP de mercado y frente al TWAP.

In [ ]:
GRID = pr.make_grid(train, n_days=3, seed=0)

class Rule:
    epsilon, gamma = 0.0, 1.0
    def __init__(self, fn, seed=0): self.fn, self.env, self.rng = fn, None, np.random.default_rng(seed)
    def act(self, s): return self.fn(self, s)
    def set_trainable(self, train=False): pass

POLICIES = {
    "TWAP determinista": TWAPAgent(),
    "aleatoria (p = ρ)": Rule(lambda a, s: a.env.action_sample(rng=a.rng)),
    "todo al principio": Rule(lambda a, s: 1),
    "todo al final": Rule(lambda a, s: int(a.env.state_pos >= a.env.episode_bins - a.env.vol_care)),
}
evals = {name: pr.evaluate(p, train, GRID) for name, p in POLICIES.items()}
pr.save_df(evals["TWAP determinista"], "00_twap_grid"); pr.save_df(evals["aleatoria (p = ρ)"], "00_random_grid")
cols = ["completed_mean", "t_fin_mean", "max_err_mean", "max_err_std", "slippage_bps_mean", "slippage_bps_std", "slippage_vs_twap_bps_mean"]
summary = pd.DataFrame({k: pr.summarize(v)[cols] for k, v in evals.items()}).T
summary.columns = ["completa", "fin (t/N)", "error máx.", "± error", "slippage (pb)", "± slippage", "vs TWAP (pb)"]
summary

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
names = list(evals)
ax[0].boxplot([evals[n].max_err for n in names], tick_labels=names); ax[0].set_title("error máx. de seguimiento (títulos)")
ax[1].boxplot([evals[n].slippage_bps for n in names], tick_labels=names); ax[1].axhline(-half, color=C_MID, ls="--", lw=1, label="-½ spread")
ax[1].set_title("slippage frente al VWAP de mercado (pb)"); ax[1].legend()
for a in ax: a.tick_params(axis="x", rotation=15)
plt.tight_layout(); pl.save(fig, "00_referencias")

In [ ]:
env = BestExecutionEnv(train, seed=0)
cfg = GRID[4]
pr.run_episode(env, TWAPAgent(env), cfg)
pl.plot_results(env, title=f"TWAP determinista · N = {cfg['n_bins']}, n = {cfg['vol_care']} · el objetivo visual del ej. 2", save="00_twap_episodio");

In [ ]:
envs = []
for cfg in GRID:
    e = BestExecutionEnv(train, seed=0); rnd = POLICIES["aleatoria (p = ρ)"]; rnd.env = e
    pr.run_episode(e, rnd, cfg); envs.append(e)
pl.plot_trajectories(envs, title="Aleatoria con p = ρ: completa la orden, pero lejos del plan", save="00_random_trayectorias");

## 8. Qué nos llevamos para los ejercicios

1. **El objetivo del ej. 2 es geométrico:** seguir la diagonal en el plano (T, V). El TWAP determinista lo hace con un error
   ≤ 1 título; ese es el listón de «comportamiento similar».
2. **El precio apenas discrimina** entre estrategias que compran agresivo (todas pagan ~½ spread). Una recompensa basada en
   el slippage global (la del notebook original) mezcla ese ruido con el objetivo: de ahí la recompensa de seguimiento del ej. 2.
3. **La aleatoria con p = ρ es el punto de partida** del aprendizaje: completa la orden pero se separa decenas de títulos del
   plan. Es la referencia E0 de la métrica de mejora del ej. 3.
4. **Los casos extremos** (todo al principio / al final) acotan el problema y muestran por qué la idea B del ej. 4 (acabar
   hacia el 90 % sin dejar pendiente) necesita calibrar bien la penalización.